In [70]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from timezonefinder import TimezoneFinder
from numpy import round


## API request

In [ ]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 10)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
lat, lng = -22.934, -43.180
tz = TimezoneFinder().certain_timezone_at(lat=lat, lng=lng)

params = {
	"latitude": lat,
	"longitude": lng,
	"hourly": ["temperature_2m", "precipitation_probability", "precipitation"],
    "forecast_hours" : 6,
	"current": ["temperature_2m", "precipitation", "wind_speed_10m", "wind_direction_10m"],
}
responses = openmeteo.weather_api(url, params=params)


In [72]:
def get_wind_direction(degrees:float) -> str:
    directions = ['N ↓', 'NE ↙', 'E ←', 'SE ↖', 'S ↑', 'SW ↗', 'W →', 'NW ↘']
    idx = round(degrees / 45) % 8
    return directions[int(idx)]

def get_lat_lng_str(coordinates:float,type:str) -> str:
    if not type in ["lat", "lng"]:
        raise ValueError("Type must be 'lat' or 'lng'")
    if type == "lat":
        return f"{abs(coordinates)}°S" if coordinates < 0 else f"{coordinates}°N"
    else:
        return f"{abs(coordinates)}°W" if coordinates < 0 else f"{coordinates}°E"

In [74]:
response = responses[0]
print(f"Coordinates: {get_lat_lng_str(response.Latitude(), 'lat')} {get_lat_lng_str(response.Longitude(), 'lng')}")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {tz}")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_temperature_2m = current.Variables(0).Value()
current_precipitation = current.Variables(1).Value()
current_wind_speed_10m = current.Variables(2).Value()
current_wind_direction_10m = current.Variables(3).Value()

print(f"\nCurrent Weather Update: { pd.to_datetime(current.Time(), unit = 's', utc = True).tz_convert(tz) }")
print(f"Current temperature_2m: {int(round(current_temperature_2m, 1))} °C")
print(f"Current precipitation: {current_precipitation} mm")
print(f"Current wind_speed_10m: {current_wind_speed_10m} m/s")
print(f"Current wind_direction_10m: {get_wind_direction(current_wind_direction_10m)} ({current_wind_direction_10m}°)")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True).tz_convert(tz),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True).tz_convert(tz),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)
    

Coordinates: 22.875°S 43.25°W
Elevation: 17.0 m asl
Timezone: America/Sao_Paulo

Current Weather Update: 2026-03-02 19:30:00-03:00
Current temperature_2m: 24 °C
Current precipitation: 0.0 mm
Current wind_speed_10m: 3.5999999046325684 m/s
Current wind_direction_10m: SE ↖ (143.13002014160156°)

Hourly data
                        date  temperature_2m  precipitation_probability  \
0 2026-03-02 19:00:00-03:00       24.633999                        0.0   
1 2026-03-02 20:00:00-03:00       24.033998                        0.0   
2 2026-03-02 21:00:00-03:00       23.584000                        0.0   
3 2026-03-02 22:00:00-03:00       23.334000                        0.0   
4 2026-03-02 23:00:00-03:00       23.084000                        0.0   
5 2026-03-03 00:00:00-03:00       22.684000                        0.0   

   precipitation  
0            0.0  
1            0.0  
2            0.0  
3            0.0  
4            0.0  
5            0.0  
